# Everstorm Data Engineer — Workshop

Build two things on Google Cloud, starting from an empty account.

| | Starts as | Ends as |
|---|---|---|
| **Part 1 · A knowledge base you query with SQL** | 180 customer-support notes written in plain English | Three relational tables, answering two business questions nobody had ever written down |
| **Part 2 · A production RAG agent** | 28 company policy documents | An AI agent on a public URL that answers customer questions from those documents, and says which one it used |

The company, **Everstorm Outfitters**, is fictional and the data is synthetic,
but the problems are real ones.

### What you'll be able to explain afterwards

1. How to query files **without loading them** (schema-on-read)
2. How to turn free text into a table with **one SQL function**, and the three prompt rules that make it reliable
3. What an **embedding** is, and why semantic search finds things keyword search can't
4. Why a real system uses **two databases**: one for analysis, one for serving
5. How a **vector index** keeps search fast as data grows
6. How a **managed pipeline** scales out and survives failures
7. What **RAG** is: retrieve, augment, generate
8. What **"production-ready"** actually requires, and how to measure whether an agent is right

### Before you start

- **Budget about 2 hours**, plus ~15 minutes to create a Google Cloud account if you don't have one.
- The run costs a few dollars of the $300 free-trial credit — **as long as you run the cleanup in §8 when you finish.** The Cloud SQL instance is the expensive part; leaving it running is what generates a surprise bill.
- Two steps have long waits built in (the database, ~10–15 min; the Dataflow job, ~3–5 min). Both are flagged where they happen.

### Services you'll use, in order

| Service | Its job |
|---|---|
| Cloud Shell | Your terminal, in the browser. Nothing to install |
| Cloud Storage | Holds the raw files |
| BigQuery | The data warehouse: query files where they sit, run Gemini from SQL, semantic search |
| Vertex AI: Gemini 2.5 Flash + `text-embedding-005` | Reads the text, and turns meaning into numbers |
| Cloud SQL for PostgreSQL + pgvector | The fast vector database the agent searches |
| Dataflow | The pipeline that fills that database |
| Cloud Build + Artifact Registry | Builds and stores the containers |
| Cloud Run | Hosts the finished agent on a URL |
| Agent Development Kit (ADK) + A2A | Builds the agent, and lets other agents call it |

## How to use this notebook

**Go top to bottom, one cell at a time.** This is a run sheet, not a Run All
notebook — several steps launch background jobs you have to wait for, and §8
deletes everything you built.

Three kinds of cell:

| Cell | What to do with it |
|---|---|
| **Code cells** (`%%bash`, `%%bigquery`) | Run them from here — if this notebook is running somewhere that has the repo at `~/everstorm-dataengineer`. Cloud Shell does. Otherwise paste the body, minus the `%%bash` line, into Cloud Shell. |
| **Code inside a markdown cell** | Paste-ready, but it runs *somewhere else*: Cloud SQL Studio (Postgres), an interactive terminal, or a file you edit in the repo. |
| **Plain markdown** | Explanation. 💡 is a concept worth keeping. ⚠️ is a mistake that costs people time. |

A few things that trip people up:

- **Every `%%bash` cell is a fresh shell.** That's why each one re-runs `set_env.sh` — nothing carries over from the cell before. It's deliberate, not redundant.
- **`%%bigquery` cells need the BigQuery magics**, loaded once at the top of Part 1.
- **Where a cell says `REPLACE-WITH-YOUR-…`**, substitute the value the previous cell printed. Those two are the only manual substitutions in the whole notebook.
- **Optional steps are marked _Optional_.** Skip them on a first run; they're worth doing on a second.

## 1. What goes where (reference)

Two corpora, two destinations, and they are **not** interchangeable. Keeping this
straight prevents most of the confusion in Part 2.

| Data | Lands in | Used by |
|---|---|---|
| `data/tickets/*.txt` — 180 support case notes | `gs://…/tickets/` | BigQuery external table, Part 1 |
| `data/kb/*.md` — 263 policy chunks | `gs://…/kb/` | Dataflow → Cloud SQL, Part 2 |

Resource names, all defined once in `set_env.sh`:

| Resource | Name |
|---|---|
| BigQuery dataset | `everstorm_data` |
| External table | `raw_tickets_table` |
| Raw model output | `structured_tickets` |
| Extracted tables | `tickets` / `orders` / `resolutions` |
| Chunk + embed tables | `chunked_tickets` / `embedded_tickets` |
| Cloud SQL instance / database | `everstorm-kb-db` / `everstorm_kb` |
| Cloud SQL vector table | `policy_chunks` (column: `chunk_content`) |
| Agent tool | `policy_lookup(question)` |
| Dataflow worker image | `everstorm-vectorizer` |
| Cloud Run service | `everstorm-support-agent` |

You never have to rename anything: the vector table name is consistent across
the DDL, the pipeline and the agent, and the pipeline reads it from a
`--kb_table` flag rather than hardcoding it.

## 2. Setup — account, project, provisioning

Six steps after the account, about 10 minutes of typing.

**Steps 1–6 run in Cloud Shell, not in this notebook.** `init.sh` stops and waits
for keyboard input, which a notebook cell can't give it.

⏳ **The database builds in the background.** Step 6 starts Cloud SQL and returns
in about a minute, but the instance takes 10–15 more minutes to finish. Part 1
doesn't need it, so you work in BigQuery while it builds and check it's ready
before Part 2. Don't sit and watch it.

### Step 0 · Google Cloud account and $300 free credit

Skip this if you already have an account with billing enabled.

**What you need:**
- **A personal Google account** (Gmail). Work and school accounts are often blocked by their organisation from creating billing accounts, and they fail halfway through.
- **A card for identity verification.** The trial doesn't charge it. Some prepaid, virtual and debit cards are rejected.
- **Never having used a Google Cloud free trial before.** It's one per person.

**Steps:**

1. Go to **https://cloud.google.com/free** → **Get started for free**
2. Sign in with a personal Google account
3. Choose your country, accept the terms, and add a payment method for verification
4. Finish. You land in the Google Cloud console, with the free-trial credit shown at the top
5. **Open Cloud Shell:** the `>_` icon, top right. The first start takes about 30 seconds. Click **Authorize** if it asks

⚠️ If you see a button saying **Activate** or **Upgrade**, leave it alone. That
converts the free trial into a paid account. You don't need it for this workshop.

**Checkpoint, in Cloud Shell:**

```bash
gcloud auth list              # your email, marked with *
gcloud billing accounts list  # one account, OPEN: True
```

### Step 1 · Get the project into Cloud Shell

Open Cloud Shell (the `>_` icon, top right of the console), then use whichever
applies to you.

**If you were given a zip file** — in the Cloud Shell window, click the **⋮**
menu (top right of the Cloud Shell panel) → **Upload** → choose the zip. It
lands in your home directory. Then:

```bash
cd ~
unzip -o everstorm-dataengineer.zip
ls ~/everstorm-dataengineer/init.sh     # must print the path, not "No such file"
```

⚠️ If that `ls` errors, the zip unpacked under a different folder name. Check
with `ls ~`, then rename it: `mv ~/<whatever-it-is> ~/everstorm-dataengineer`.
**Every command in this notebook assumes the path `~/everstorm-dataengineer`.**

**Or clone it from GitHub** — the repo is public, so no login is needed:

```bash
git clone https://github.com/kezhen-yang/everstorm-dataengineer.git ~/everstorm-dataengineer
```

Either way, confirm you're authenticated before moving on:

```bash
gcloud auth list     # your email, marked with *
```

### Step 2 · Project and billing

```bash
chmod +x ~/everstorm-dataengineer/*.sh
cd ~/everstorm-dataengineer && ./init.sh
gcloud config set project $(cat ~/project_id.txt) --quiet
```

`init.sh` creates a project, or reuses one if `~/project_id.txt` already names
it. It asks for an ID, with a random suggestion pre-filled. Then it links your
first open billing account.

⚠️ **Check that billing actually linked.** `init.sh` prints `Full Setup Complete`
even when the billing step failed, because the billing script exits normally
after printing its error. Don't trust that line. Trust this one:

```bash
gcloud billing projects describe $(cat ~/project_id.txt) --format="value(billingEnabled)"   # True
```

If it prints anything other than `True`, stop and fix it here — every step after
this one depends on billing. See the troubleshooting table at the end of §2.

### Step 3 · Enable the APIs

```bash
gcloud services enable storage.googleapis.com bigquery.googleapis.com \
  sqladmin.googleapis.com aiplatform.googleapis.com dataflow.googleapis.com \
  pubsub.googleapis.com cloudfunctions.googleapis.com run.googleapis.com \
  cloudbuild.googleapis.com artifactregistry.googleapis.com iam.googleapis.com \
  compute.googleapis.com cloudresourcemanager.googleapis.com \
  cloudaicompanion.googleapis.com bigqueryunified.googleapis.com
```

💡 **That list is the architecture diagram in disguise.** Storage: where the raw
files live. BigQuery: the warehouse for Part 1. SQL Admin: the Postgres database
for Part 2. AI Platform: that's Vertex, where Gemini and the embedding model
live. Dataflow: the pipeline that loads the database. Cloud Build and Artifact
Registry: building and storing containers. Run: hosting the agent at the end.

Every service in Google Cloud starts switched **off**. Nothing runs just because
the project exists — you turn on exactly what you use, and you're billed for
exactly what you turn on.

### Step 4 · Environment and container registry

```bash
source ~/everstorm-dataengineer/set_env.sh
echo $SERVICE_ACCOUNT_NAME      # must end in -compute@developer.gserviceaccount.com

gcloud artifacts repositories create $REPO_NAME \
  --repository-format=docker --location=$REGION \
  --description="Everstorm workshop images"
```

💡 **`source`, not `./`.** One file, `set_env.sh`, holds every name in this
workshop: the bucket, the database, the region, the service. Everything else
reads from it, so nothing is typed twice. If you run it with `./set_env.sh`
instead of `source`, it executes in a separate shell, sets all those variables,
and then that shell exits and takes them with it. Your terminal never sees them,
and every command afterwards fails on blank names. If something later complains
about an empty bucket name, this is why.

⚠️ **If `$SERVICE_ACCOUNT_NAME` is blank**, the Compute API is still switching
on. Wait 30 seconds and `source` again. **Don't continue with it blank** — the
next step would grant roles to nobody, and report no error while doing it.

### Step 5 · Permissions

Still in the same Cloud Shell session, so the variables from Step 4 are set:

```bash
for ROLE in storage.admin bigquery.admin dataflow.admin cloudsql.admin \
            pubsub.admin aiplatform.user cloudbuild.builds.editor \
            artifactregistry.admin run.admin iam.serviceAccountUser \
            logging.logWriter; do
  gcloud projects add-iam-policy-binding $PROJECT_ID \
    --member="serviceAccount:$SERVICE_ACCOUNT_NAME" --role="roles/$ROLE" --quiet >/dev/null \
    && echo "granted $ROLE"
done
```

Expect eleven `granted …` lines.

💡 **What that loop did.** It took the *default service account* — a robot
identity every project comes with — and gave it the roles it needs to run the
pipeline and the agent on your behalf.

These are deliberately broad roles: storage *admin*, BigQuery *admin*, SQL
*admin*. For a workshop that's the right trade, because debugging a permission
error halfway through the build is miserable. In production you'd give each workload its
own service account with the narrowest roles that work.

Hold on to the idea of identities — there are **two more** coming. BigQuery needs
one of its own in Part 1, and the agent on Cloud Run needs one in Part 2. That's
where people usually get stuck.

### Step 6 · Provision and upload

In [ ]:
%%bash
cd ~/everstorm-dataengineer
source ~/everstorm-dataengineer/set_env.sh
./data_setup.sh     # Cloud SQL (~15 min, background) + bucket + BOTH uploads

`data_setup.sh` does three things: starts creating the Postgres database **in
the background** (the 15-minute part), creates the storage bucket, and uploads
both datasets — `data/tickets/` → `gs://…/tickets/` and `data/kb/` →
`gs://…/kb/`. There is nothing to generate and nothing to copy by hand; the
flattened tickets and the 263 chunks are already in the repo.

💡 **It's safe to run twice.** Before creating anything it checks whether the
database and bucket already exist. So if it dies halfway through — which on a
real network happens — you just run it again and it carries on. That property is
called **idempotence**, and it's worth building into every setup script you
write.

⚠️ **A failed database create is silent**, because the script sends `gcloud sql`
output to `/dev/null`. Confirm it actually started:

```bash
gcloud sql instances list      # everstorm-kb-db → PENDING_CREATE
```

### Verification gate

Don't move on until this passes.

In [ ]:
%%bash
. ~/everstorm-dataengineer/set_env.sh
gcloud storage ls gs://${BUCKET_NAME}/tickets/ | wc -l   # expect 180
gcloud storage ls gs://${BUCKET_NAME}/kb/      | wc -l   # expect 263
gcloud sql instances describe $INSTANCE_NAME --format="value(state)"  # PENDING_CREATE is fine here

You want **180** and **263**.

The third line will probably say `PENDING_CREATE` — the database is still being
built, which is expected and fine. Part 1 doesn't touch it, and you check it
again at the bridge before Part 2.

If the first two numbers are wrong, re-run `./data_setup.sh` before continuing.

#### Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Card declined during sign-up | Some prepaid, virtual and debit cards are rejected for verification | Try another card |
| *Not eligible* for the free trial | That Google account has had a free trial, or been a paying customer, before | Use a different personal Google account |
| Sign-up or billing blocked | Work or school Google account restricted by its organisation | Use a personal Gmail |
| `init.sh`: *exceeded your allotted project quota*, then keeps asking for another ID | The account's project quota is full, so **every** ID fails | Ctrl+C. Use an existing project: `echo <id> > ~/project_id.txt`, then rerun `./init.sh`. Deleting projects frees nothing for 30 days |
| `init.sh` prints *Full Setup Complete* right after a billing error | The billing script exits normally even on failure | Check `gcloud billing projects describe <id> --format="value(billingEnabled)"` |
| Billing: *Project … has been deleted*, right after an undelete | Propagation lag | Wait 1–2 minutes, then retry |
| `billing projects link`: *Cloud billing quota exceeded* | The billing account already has its maximum number of linked projects | `gcloud billing projects list --billing-account=<id>`, then `gcloud billing projects unlink <unused-project>`, then link again |
| `$SERVICE_ACCOUNT_NAME` or `$BUCKET_NAME` is blank | Ran `./set_env.sh` instead of `source`, or the Compute API is still enabling | `source ./set_env.sh` again |
| `gcloud sql instances list` shows nothing after `data_setup.sh` | The create failed silently (output goes to `/dev/null`) | Rerun `./data_setup.sh`. It's idempotent |

## 3. Part 1 — Knowledge base with SQL (BigQuery + Gemini)

Load the BigQuery magics once, before anything else. Harmless if they're already
loaded (BigQuery Studio notebooks preload them); run
`pip install bigquery-magics` first if they aren't installed.

In [ ]:
%load_ext bigquery_magics

### Where you're starting

Open one of the files in `data/tickets/` and read it.

It's a customer-support case note — a human being typed it, in prose, at the end
of a phone call. There are 180 of them.

Everything you'd want to know is *in* there: which product, which carrier, how
long it took to close, whether you refunded. But it's in there as **English**.
You can't sort it. You can't `GROUP BY` it. You can't put it on a dashboard. If
your director asks "which of our warehouses is slowest", the honest answer today
is: someone reads 180 of these and makes a tally.

In the next forty minutes they become a table you can query in SQL. Not by
writing a parser — by asking a model to read them.

**Before you run anything**, read one ticket and write down the columns you'd
want if you were building this table by hand. You'll end up with almost exactly
the schema this notebook uses. That's the point: deciding the schema was never
the hard part. Doing it 180 times was.

### Two questions to answer by the end of Part 1

1. **Which fulfilment centre is slowest to resolve a ticket?**
2. **Which product generates the most warranty claims?**

Nobody at this fictional company has ever written either of those down. They're
sitting latent in 180 pieces of prose.

### Connection, dataset, permissions

In [ ]:
%%bash
. ~/everstorm-dataengineer/set_env.sh

bq mk --connection --connection_type=CLOUD_RESOURCE \
  --project_id=${PROJECT_ID} --location=${REGION} gcs-connection

bq --location=${REGION} mk --dataset ${PROJECT_ID}:everstorm_data

export CONNECTION_SA=$(bq show --connection --project_id=${PROJECT_ID} \
  --location=${REGION} --format=json gcs-connection | jq -r '.cloudResource.serviceAccountId')
echo "Connection SA: $CONNECTION_SA"

gcloud storage buckets add-iam-policy-binding gs://${BUCKET_NAME} \
  --member="serviceAccount:$CONNECTION_SA" --role="roles/storage.objectViewer"
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
  --member="serviceAccount:$CONNECTION_SA" --role="roles/aiplatform.user"

echo "${PROJECT_ID}.${REGION}.gcs-connection"   # copy this
echo $BUCKET_NAME                               # and this

📋 **Copy the last two lines somewhere.** You need them for the two
`REPLACE-WITH-YOUR-…` markers coming up.

💡 **Why the connection exists.** BigQuery, by default, cannot reach your storage
bucket and cannot call Gemini — it has no identity of its own for that. The first
command created a **connection resource**, and behind it Google automatically
minted a service account. That's the identity BigQuery wears when it goes and
reads your files or calls a model. So you have to grant *that* account two
things: read access on the bucket, and permission to call Vertex AI. That's
exactly what the two `add-iam-policy-binding` commands do.

⚠️ **There is a second identity in this project.** Later, when the agent runs on
Cloud Run, that's a *different* service account, and it needs its own permission
to call Vertex. Two identities: the one BigQuery wears at build time, and the one
the agent wears at query time. A 403 in Part 2 is almost always because you
granted the first and forgot the second.

### External table — schema-on-read

Replace `REPLACE-WITH-YOUR-BUCKET` with the bucket name the previous cell
printed.

In [ ]:
%%bigquery
CREATE OR REPLACE EXTERNAL TABLE everstorm_data.raw_tickets_table (
  raw_text STRING
)
OPTIONS (
  format = 'CSV',
  field_delimiter = '§',
  quote = '',
  uris = ['gs://REPLACE-WITH-YOUR-BUCKET/tickets/*']
);

In [ ]:
%%bigquery
SELECT COUNT(*) FROM everstorm_data.raw_tickets_table;   -- expect 180

In [ ]:
%%bigquery
SELECT * FROM everstorm_data.raw_tickets_table LIMIT 3;

💡 **Notice what did *not* just happen.** You didn't load anything. There was no
import job. There's no copy of these tickets inside BigQuery now. That bucket is
still the only place this data exists — BigQuery just read it, live, when you hit
run.

That's **schema-on-read**: you declared a shape (one column, `raw_text`) and
BigQuery applied that shape to the files at query time. Which means there's no
dev copy drifting away from the prod copy, and no nightly sync to keep green.
One copy of the truth, in the bucket, and everything reads from it.

💡 **The `§` delimiter looks like a hack, and it's a deliberate one.** You told
BigQuery this is a CSV. It isn't — it's prose. But the field delimiter is set to
a section sign, a character that never appears anywhere in this text. So BigQuery
goes looking for a column separator, never finds one, and the entire line lands
in a single string column. Which is exactly what you want.

Leave the default comma and every comma inside a sentence splits a new column,
shredding the whole thing. `quote = ''` is there for the same reason — otherwise
apostrophes and quote marks in ordinary English start opening and closing CSV
fields.

⚠️ **The one to actually remember: this table reads one row per _line_.** Not one
row per file. One row per line.

That's fine if every source file is a single line of text. These tickets
originally weren't — they were wrapped at 86 characters, like a normal document,
about 17 lines each. Pointed at those, this table returns about **3,100 rows**,
not 180. Every line of every ticket becomes its own row, half a sentence at a
time. And then you hand *that* to a model and ask it to extract a product name
from `delivered 27 Aug to Halifax NS. That puts the request on`.

It doesn't error. That's the dangerous part. It runs fine and produces garbage.

So there's a preprocessing step that flattens each ticket to one line before
upload. That's what's in `data/tickets/` — the flattened version is what BigQuery
sees.

💡 **The general lesson, bigger than this command: every pipeline carries an
assumption about the shape of its input.** This one assumes one record per line.
It doesn't check, it doesn't warn, it just quietly means something different.
You'll hit two more of these today.

**_Optional:_** point `uris` at the unflattened source and watch the count come
back ~3,100 before fixing it. Seeing the wrong number is worth more than reading
about it.

### `ML.GENERATE_TEXT` — the extraction

Replace `REPLACE-WITH-YOUR-FULL-CONNECTION-STRING` with the connection string
printed earlier — it looks like `your-project.us-central1.gcs-connection`. Keep
the backticks.

In [ ]:
%%bigquery
CREATE OR REPLACE MODEL everstorm_data.gemini_flash_model
REMOTE WITH CONNECTION `REPLACE-WITH-YOUR-FULL-CONNECTION-STRING`
OPTIONS (endpoint = 'gemini-2.5-flash');

**Read the prompt in the next cell before you run it.** The SQL is one function
call wrapped around a `SELECT` — there's no Python service, no Dataflow job, no
regex. All the engineering is in the prompt, and the three rules below are what
decide whether you get a usable table or a mess.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE everstorm_data.structured_tickets AS
SELECT ml_generate_text_result AS structured_data
FROM ML.GENERATE_TEXT(
  MODEL everstorm_data.gemini_flash_model,
  (
    SELECT CONCAT(
      """
      From the following customer-support case note, extract structured data
      into a single, valid JSON object.

      Your output must strictly conform to this structure and these data types.
      Do not add, remove, or rename any keys.

      {
        "ticket": {
          "ticket_id": "string",
          "opened_date": "string (YYYY-MM-DD)",
          "closed_date": "string (YYYY-MM-DD, or null if never stated)",
          "channel": "string",
          "agent": "string",
          "issue_category": "string",
          "csat": "integer or null"
        },
        "order": {
          "ticket_id": "string",
          "order_id": "string or null",
          "product_name": "string or null",
          "region": "string",
          "fulfillment_center": "string or null",
          "carrier": "string or null",
          "payment_method": "string or null"
        },
        "resolution": {
          "ticket_id": "string",
          "outcome": "string",
          "refund_amount_usd": "number or null"
        }
      }

      **CONSTRAINED FIELDS — use exactly one of the listed values:**
      - issue_category: shipping_delay, return_request, warranty_claim,
        exchange_sizing, refund_status, duties_customs, lost_parcel,
        damaged_in_transit, product_question, payment_declined,
        address_change, order_cancellation
      - region: US West, US Midwest, US East, Canada, EU, UK, AU/NZ, Japan/SG
      - fulfillment_center: Reno NV, Harrisburg PA, Rotterdam NL
      - outcome: resolved, refunded, replaced, escalated, closed_no_action

      **CRUCIAL RULES:**
      - issue_category is never stated literally. Classify it from the content.
      - region is often not stated. Infer it from the destination city.
      - Use null where a value is genuinely absent. Never invent a value and
        never substitute 0 for a missing number.
      - Output ONLY the raw JSON object. No markdown fences, no commentary.

      Here is the case note:
      """,
      raw_text
    ) AS prompt
    FROM everstorm_data.raw_tickets_table
  ),
  STRUCT(0.2 AS temperature, 2048 AS max_output_tokens)
);

### The three prompt rules — the transferable part of Part 1

💡 **Rule 1 — constrain the label space.** Look at `issue_category`: twelve
values, listed explicitly.

Here's the thing: **`issue_category` is never written anywhere in the ticket.** Go
read one. Nobody typed `shipping_delay`. The agent wrote a paragraph about a
parcel being late. So the model isn't *extracting* this field — it's
*classifying*, which is a genuinely different and harder job.

Without that list it will still answer. It'll give you `shipping_issue` on one
ticket, `late_delivery` on the next, `Shipping Delay` with a capital S on the
third, `delivery_problem` on the fourth. All four are *correct*. And your
`GROUP BY` is now garbage, because those are four buckets that should have been
one.

Any time you ask a model for a category, hand it the categories. This is the
single highest-value line in the prompt.

💡 **Rule 2 — extract facts, compute derivations.** The prompt asks for
`closed_date`. It does **not** ask how many days the ticket took.

Why: the ticket says "Closed 16 Jan". That's a fact, sitting right there in the
text — the model just has to see it. "It took four days" is *not* in the text;
that's arithmetic. Ask a model for arithmetic and sometimes you get four,
sometimes five, and you have no way of knowing which.

So ask the model for what's literally in the document, and do the derivation in
SQL — where it's deterministic and testable. Don't make a language model your
calculator.

💡 **Rule 3 — say the word `null` out loud.** Note the instruction: *use null
where a value is genuinely absent, never substitute zero.*

About a third of these tickets have no satisfaction score — the customer never
filled in the survey. Plenty have no refund, because nothing was refunded.

Without that instruction, models are *helpful*. They'll give you a zero. It looks
like data. It sits in the column looking perfectly reasonable. And now your
average CSAT includes a pile of fake zeros, it's wrong, and **nothing anywhere
will tell you it's wrong.** There's no error — just a number on a dashboard
that's lower than reality.

In [ ]:
%%bigquery
SELECT * FROM everstorm_data.structured_tickets LIMIT 1;

💡 **Did you get your table? No — not even slightly.**

What came back is the model's *response envelope* — `candidates`, `content`,
`parts`, `text` — and somewhere down inside it your JSON is sitting there as a
**string**. Not as structure. As text, inside a field, inside an envelope.

This is true of every LLM-in-SQL function on every platform, and it's the bit the
demos skip. Getting the model to answer was the easy half. Now you have to dig
the answer out and make it relational.

### Normalise to three tables

Each of these three cells pulls the JSON text out of the envelope, parses it, and
flattens one section into a proper table.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE everstorm_data.tickets AS
WITH Cleaned AS (
  SELECT SAFE.PARSE_JSON(
    REGEXP_EXTRACT(
      JSON_VALUE(structured_data, '$.candidates[0].content.parts[0].text'),
      r'\{[\s\S]*\}')
  ) AS d
  FROM everstorm_data.structured_tickets
)
SELECT
  JSON_VALUE(d, '$.ticket.ticket_id')            AS ticket_id,
  SAFE_CAST(JSON_VALUE(d, '$.ticket.opened_date') AS DATE) AS opened_date,
  SAFE_CAST(JSON_VALUE(d, '$.ticket.closed_date') AS DATE) AS closed_date,
  JSON_VALUE(d, '$.ticket.channel')              AS channel,
  JSON_VALUE(d, '$.ticket.agent')                AS agent,
  JSON_VALUE(d, '$.ticket.issue_category')       AS issue_category,
  SAFE_CAST(JSON_VALUE(d, '$.ticket.csat') AS INT64) AS csat
FROM Cleaned
WHERE d IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY ticket_id ORDER BY opened_date) = 1;

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE everstorm_data.orders AS
WITH Cleaned AS (
  SELECT SAFE.PARSE_JSON(
    REGEXP_EXTRACT(
      JSON_VALUE(structured_data, '$.candidates[0].content.parts[0].text'),
      r'\{[\s\S]*\}')
  ) AS d
  FROM everstorm_data.structured_tickets
)
SELECT
  JSON_VALUE(d, '$.order.ticket_id')          AS ticket_id,
  JSON_VALUE(d, '$.order.order_id')           AS order_id,
  JSON_VALUE(d, '$.order.product_name')       AS product_name,
  JSON_VALUE(d, '$.order.region')             AS region,
  JSON_VALUE(d, '$.order.fulfillment_center') AS fulfillment_center,
  JSON_VALUE(d, '$.order.carrier')            AS carrier,
  JSON_VALUE(d, '$.order.payment_method')     AS payment_method
FROM Cleaned
WHERE d IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY ticket_id ORDER BY order_id) = 1;

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE everstorm_data.resolutions AS
WITH Cleaned AS (
  SELECT SAFE.PARSE_JSON(
    REGEXP_EXTRACT(
      JSON_VALUE(structured_data, '$.candidates[0].content.parts[0].text'),
      r'\{[\s\S]*\}')
  ) AS d
  FROM everstorm_data.structured_tickets
)
SELECT
  JSON_VALUE(d, '$.resolution.ticket_id') AS ticket_id,
  JSON_VALUE(d, '$.resolution.outcome')   AS outcome,
  SAFE_CAST(JSON_VALUE(d, '$.resolution.refund_amount_usd') AS FLOAT64) AS refund_amount_usd
FROM Cleaned
WHERE d IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY ticket_id ORDER BY outcome) = 1;

### The payoff — question 1: which fulfilment centre is slowest?

In [ ]:
%%bigquery
SELECT
  o.fulfillment_center,
  COUNT(*) AS tickets,
  ROUND(AVG(DATE_DIFF(t.closed_date, t.opened_date, DAY)), 1) AS avg_days_to_close
FROM everstorm_data.tickets t
JOIN everstorm_data.orders o USING (ticket_id)
WHERE t.closed_date IS NOT NULL AND o.fulfillment_center IS NOT NULL
GROUP BY 1
ORDER BY avg_days_to_close DESC;

💡 **Be precise about what just happened**, because it's easy to undersell.
Nobody at this company wrote that ranking down. It's not in a report. No analyst
produced it. It was *latent* — smeared across 180 paragraphs of English that a
human would have to read end to end to notice. And you got it with a `GROUP BY`.

⚠️ **One honest footnote:** this is **calendar** days, because `DATE_DIFF` counts
calendar days. The reference figures in `data/tickets_ground_truth.csv` are
**business** days. So the absolute numbers differ and the ranking doesn't — a
decent reminder that a metric without its definition attached is just a number.
"Average days to close" means two different things depending on who built it.

### Question 2 — warranty claims, the three-table join

In [ ]:
%%bigquery
SELECT
  o.product_name,
  COUNT(*) AS warranty_claims,
  COUNTIF(r.outcome = 'replaced') AS replaced
FROM everstorm_data.tickets t
JOIN everstorm_data.orders o      USING (ticket_id)
JOIN everstorm_data.resolutions r USING (ticket_id)
WHERE t.issue_category = 'warranty_claim'
GROUP BY 1
ORDER BY warranty_claims DESC
LIMIT 5;

💡 One product should be well clear at the top. Go and read a couple of its
tickets — the failure mode is the same sentence every time.

That's a product-recall conversation. A manufacturing-lot conversation. And it
was invisible — not hidden, nobody was concealing it — just spread thin across
180 paragraphs that no single person ever read all of.

Three tables. One join. That's Part 1.

*(Yes, the finding is planted in the data generator. Real corpora contain
findings like it; a synthetic one only does if you put it there, and a demo where
every aggregate comes back flat teaches nothing.)*

### _Optional_ — measure the extraction against ground truth

The repo ships `data/tickets_ground_truth.csv`: what the generator actually
wrote, before any model read it. Load it and diff.

In [ ]:
%%bash
. ~/everstorm-dataengineer/set_env.sh
bq load --autodetect --replace --source_format=CSV \
  ${PROJECT_ID}:everstorm_data.ground_truth \
  ~/everstorm-dataengineer/data/tickets_ground_truth.csv

In [ ]:
%%bigquery
SELECT
  COUNTIF(t.issue_category = g.issue_category) / COUNT(*) AS category_accuracy,
  COUNTIF(t.closed_date    = g.closed_date)    / COUNT(*) AS date_accuracy
FROM everstorm_data.tickets t
JOIN everstorm_data.ground_truth g USING (ticket_id);

Extraction will not be 100%. Knowing the size of that gap is more useful than a
demo that pretends it doesn't exist — and most tutorials skip it.

### Chunk → embed → semantic search, all inside BigQuery

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE everstorm_data.chunked_tickets AS
WITH Numbered AS (
  SELECT ROW_NUMBER() OVER () AS doc_id, raw_text
  FROM everstorm_data.raw_tickets_table
)
SELECT
  doc_id,
  CONCAT(CAST(doc_id AS STRING), '-',
         CAST(ROW_NUMBER() OVER (PARTITION BY doc_id) AS STRING)) AS chunk_id,
  TRIM(chunk) AS chunk_text
FROM Numbered, UNNEST(SPLIT(raw_text, '.')) AS chunk
WHERE LENGTH(TRIM(chunk)) > 15;

Same connection string as before.

In [ ]:
%%bigquery
CREATE OR REPLACE MODEL everstorm_data.text_embedding_model
REMOTE WITH CONNECTION `REPLACE-WITH-YOUR-FULL-CONNECTION-STRING`
OPTIONS (endpoint = 'text-embedding-005');

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE everstorm_data.embedded_tickets AS
SELECT * FROM ML.GENERATE_EMBEDDING(
  MODEL everstorm_data.text_embedding_model,
  (SELECT doc_id, chunk_id, chunk_text AS content FROM everstorm_data.chunked_tickets),
  STRUCT('RETRIEVAL_DOCUMENT' AS task_type)
);

**Run the keyword search first.** The contrast is the whole lesson, and it only
works in this order.

In [ ]:
%%bigquery
SELECT chunk_text FROM everstorm_data.chunked_tickets
WHERE LOWER(chunk_text) LIKE '%fell apart%';   -- zero rows

Zero rows. And you *know* that's wrong — you read those warranty tickets ninety
seconds ago. They're in there. The database is telling you they don't exist
because nobody used your words.

Same question, different mechanism:

In [ ]:
%%bigquery
SELECT base.content, distance
FROM VECTOR_SEARCH(
  TABLE everstorm_data.embedded_tickets,
  'ml_generate_embedding_result',
  (SELECT ml_generate_embedding_result
   FROM ML.GENERATE_EMBEDDING(
     MODEL everstorm_data.text_embedding_model,
     (SELECT 'customers whose boots fell apart' AS content),
     STRUCT('RETRIEVAL_QUERY' AS task_type))),
  top_k => 5,
  distance_type => 'COSINE');

💡 **Read the top result against your query.** Something like *"the outsole has
separated from the midsole across the forefoot"* versus *"customers whose boots
fell apart"* — not one word in common. No "boots". No "fell". No "apart". And
it's obviously the right answer.

That's the whole idea. Every chunk was converted into a vector — a list of 768
numbers that encodes roughly what it *means*. Your question was converted into a
vector the same way. Then you just asked: which of these is closest?

**Meaning, not spelling.** That's the thing keyword search fundamentally cannot
do, and it's the engine under everything in Part 2.

⚠️ **One detail that pays off in Part 2.** The *documents* were embedded with
task type `RETRIEVAL_DOCUMENT`. The *question* was embedded with
`RETRIEVAL_QUERY`. Those are deliberately different.

A question and an answer don't look alike — "how long do I have to return this"
and "the return window is thirty days" are different shapes of text. So the model
embeds them with slightly different strategies so they land near each other
anyway.

Remember that. The agent code you're handed in Part 2 gets it wrong, and you're
going to fix it.

### Bridge to Part 2 — and start two things building

💡 **You just built a complete RAG data pipeline: chunk, embed, search. And it
never left BigQuery.** No vector database, no extra infrastructure, nothing to
operate. So why isn't that the end?

**Latency.** BigQuery is an *analytical* warehouse. It's built to scan enormous
amounts of data, it's extremely good at that, and it takes a second or two to
answer. Completely fine when you're an analyst running a `GROUP BY`. Not fine
when a customer is sitting in a chat window waiting.

So in Part 2 the knowledge moves into an *operational* database — Cloud SQL,
Postgres — where a lookup comes back in milliseconds. Same three ideas: chunk,
embed, search. Different engine, because it's a different job.

**Analytical for building the knowledge. Operational for serving it.**

⏳ **Start the pipeline worker image building now.** It takes 5–8 minutes and Part
2 needs it, so kick it off before you read any further.

In [ ]:
%%bash --bg
. ~/everstorm-dataengineer/set_env.sh
cd ~/everstorm-dataengineer/pipeline
gcloud builds submit --config cloudbuild.yaml \
  --substitutions=_REGION=${REGION},_REPO_NAME=${REPO_NAME} .

**Now check the database.** It started building at the end of setup, so it should
be `RUNNABLE` by now. If it still says `PENDING_CREATE`, read the pgvector
concepts below and run the DDL once it flips.

In [ ]:
%%bash
. ~/everstorm-dataengineer/set_env.sh
gcloud sql instances describe $INSTANCE_NAME --format="value(state)"   # RUNNABLE

## 4. Part 2 — Production RAG (Cloud SQL + Dataflow + agent)

### Cloud SQL + pgvector + HNSW

In [ ]:
%%bash
. ~/everstorm-dataengineer/set_env.sh
gcloud sql databases create $DB_NAME --instance=$INSTANCE_NAME

SERVICE_ACCOUNT_EMAIL=$(gcloud sql instances describe $INSTANCE_NAME \
  --format="value(serviceAccountEmailAddress)")
gcloud projects add-iam-policy-binding $PROJECT_ID \
  --member="serviceAccount:$SERVICE_ACCOUNT_EMAIL" --role="roles/aiplatform.user"

Now open **Cloud SQL Studio** in the console (Cloud SQL → your instance → Cloud
SQL Studio), sign in to the `everstorm_kb` database as user `postgres`, and run:

```sql
CREATE EXTENSION IF NOT EXISTS vector;
CREATE EXTENSION IF NOT EXISTS google_ml_integration CASCADE;

CREATE TABLE policy_chunks (
  id SERIAL PRIMARY KEY,
  chunk_content TEXT,
  embedding VECTOR(768)
);
```

💡 **Three columns: an id, the text, and a vector of 768 numbers.**

That 768 is not a preference. It's the output size of `text-embedding-005`, the
model used all the way through this workshop. If those two numbers disagree,
nothing works — and you find out on *insert*, not on create.

You also can't change it later. Once there's data in this column the width is
fixed; switching embedding models means a new column and re-embedding
everything. Worth ten seconds of thought before you type it.

Prove the round-trip by hand before automating it — still in Cloud SQL Studio:

```sql
SET session.my_search_var = 'Everstorm returns: 30 days from delivery, extended to 31 January for purchases made 1 November to 31 December.';

INSERT INTO policy_chunks (chunk_content, embedding)
VALUES (current_setting('session.my_search_var'),
        (embedding('text-embedding-005', current_setting('session.my_search_var')))::vector);

SELECT id, LEFT(chunk_content, 60), LEFT(embedding::TEXT, 60) FROM policy_chunks;

CREATE INDEX ON policy_chunks USING hnsw (embedding vector_cosine_ops);
```

💡 **Look at what that `INSERT` did.** Postgres called out to Vertex AI, from
inside a SQL statement, got a vector back, and stored it. No application code —
the database did it. Now you know the plumbing works, so when the pipeline
misbehaves in ten minutes you already know it isn't this.

💡 **Why the HNSW index.** Without an index, a similarity search reads every
single row and computes a distance against each one. At 200 rows, who cares. At 2
million, that's your afternoon.

HNSW builds a layered graph. Think of navigating to an address: the top layer is
the motorway network, very sparse, gets you to roughly the right city in a few
hops. Then you drop to a denser layer for the right neighbourhood. Then the
street. You never look at every house.

⚠️ **The index operator class and the query operator must match.** The index
above specifies `vector_cosine_ops`, and the queries use the cosine operator
(`<=>`). If you index with cosine and then query with Euclidean distance,
Postgres **doesn't error** — it silently ignores your index and scans everything.
Your query still returns correct results. It's just mysteriously slow forever.

### ⚠️ Look hard at what this table *doesn't* have

No document id. No revision number. No status column. No region. It's text and a
vector, and that's it.

That's a problem, because the knowledge base deliberately contains a
**superseded** shipping policy — an old revision saying free shipping starts at
$99, sitting right next to the current one saying $75. With this schema you
cannot filter on status, because there *is* no status.

So the workaround — and it **is** a workaround — is that when those documents
were chunked, a line of context was prepended into the text itself. Every chunk
literally begins with something like
`Everstorm Outfitters, Shipping and Delivery Policy, Revision 4.0, Status: current`.

Which means the version information survives into what the model reads, even
though the database knows nothing about it. It's carried in the **payload**
instead of the **schema**.

In a real build you'd add those as actual columns and filter on them in the
`WHERE` clause. Here you'll make the *model* handle it instead — and later you'll
see that work, and also see why the column would be better.

### The Dataflow pipeline

Set up the Python environment first.

In [ ]:
%%bash
cd ~/everstorm-dataengineer
. ~/everstorm-dataengineer/set_env.sh
python -m venv env
source ~/everstorm-dataengineer/env/bin/activate
cd ~/everstorm-dataengineer/pipeline
pip install -r requirements.txt

**Fill the five `#REPLACE` markers in `pipeline/vectorize_kb_pipeline.py`.** Open
it in the Cloud Shell editor. A completed reference copy is in
`solutions/pipeline_completed.py` if you get stuck — try it yourself first.

| Marker | Code |
|---|---|
| `#REPLACE-EMBEDDING-LOGIC` | `client.models.embed_content(model="text-embedding-005", contents=contents, config=EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT", output_dimensionality=768))` |
| `#REPLACE ME-READFILE` | `MatchFiles` → `ReadMatches` → `beam.Map(lambda f: (f.metadata.path, f.read_utf8()))` |
| `#REPLACE ME-EMBEDDING` | `beam.BatchElements(min_batch_size=1, max_batch_size=2)` → `beam.ParDo(EmbedTextBatch(...)).with_outputs('failed', main='processed')` |
| `#REPLACE ME-WRITE TO DB` | `beam.ParDo(WriteEssenceToSpellbook(...))` on `embeddings.processed` |
| `#REPLACE ME-LOG FAILURES` | `beam.Map(...)` logging on `embeddings.failed` |

**_Optional but recommended:_** validate locally with `DirectRunner` first. It's
cheap, and it catches every credential and schema error before you spend Dataflow
worker minutes. Skip it if you're short on time and go straight to the cell after.

In [ ]:
%%bash
. ~/everstorm-dataengineer/set_env.sh
source ~/everstorm-dataengineer/env/bin/activate
cd ~/everstorm-dataengineer/pipeline

python3 vectorize_kb_pipeline.py \
  --runner=DirectRunner \
  --project=$PROJECT_ID --region=$REGION \
  --job_name="everstorm-local-test-$(date +%Y%m%d-%H%M%S)" \
  --temp_location="gs://${BUCKET_NAME}/dataflow/temp" \
  --staging_location="gs://${BUCKET_NAME}/dataflow/staging" \
  --input_pattern="gs://${BUCKET_NAME}/kb/*.md" \
  --instance_name=$INSTANCE_NAME

Then at scale, on Dataflow. **This needs the container image from the bridge
build — make sure that finished.**

In [ ]:
%%bash
. ~/everstorm-dataengineer/set_env.sh
source ~/everstorm-dataengineer/env/bin/activate
cd ~/everstorm-dataengineer/pipeline

python vectorize_kb_pipeline.py \
  --runner=DataflowRunner \
  --project=$PROJECT_ID --job_name=$DF_JOB_NAME \
  --temp_location="gs://${BUCKET_NAME}/dataflow/temp" \
  --staging_location="gs://${BUCKET_NAME}/dataflow/staging" \
  --sdk_container_image="${REGION}-docker.pkg.dev/${PROJECT_ID}/${REPO_NAME}/everstorm-vectorizer:latest" \
  --sdk_location=container --experiments=use_runner_v2 \
  --input_pattern="gs://${BUCKET_NAME}/kb/*.md" \
  --instance_name=$INSTANCE_NAME --region=$REGION

⏳ **The job takes 3–5 minutes to spin up.** Watch it in the console under
Dataflow → Jobs. While you wait, three things worth understanding.

💡 **What you didn't write.** Look back at the pipeline: read files, batch them,
embed the batch, write to Postgres. Four steps, maybe forty lines.

What you did *not* write: anything about machines. No worker pool. No "if the API
rate-limits, back off and retry". No "if a worker dies, redistribute its work". No
shutdown logic.

That's the split. **Beam** is where you describe *what should happen* — the
logical shape of the work. **Dataflow** decides how many machines to start, what
each one does, what to do when one falls over, and when to turn everything off.
If this were a Python script on your laptop, everything in that second list would
be your problem.

💡 **The failure branch.** Notice the pipeline has two outputs — `processed` and
`failed`. If a batch fails to embed (bad character, model timeout, whatever) it
doesn't crash the job. It gets tagged, routed to a side output, and logged. The
other 260 keep going.

Ask what a hand-written loop does at file 200 of 263. It throws. You've got 60% of
your data in the database, no record of *which* 60%, and your only move is to
truncate and start over. The side output is the difference between a script and a
pipeline, and it's about fifteen lines.

⚠️ **The chunking gap — the one to take home.** Go and look at this pipeline for a
chunking step. Read the whole thing. **There isn't one.** It reads a file, embeds
*the entire file* as one vector, and writes it. One file, one vector.

That's survivable if your documents are tiny. These policy documents are 800–1,300
words. Take 1,300 words about shipping — rates, transit times, holiday cut-offs,
lost parcels — and average all of it into a single vector, and you get a vector
that means "shipping, generally". It's mush. It matches every shipping question
equally badly.

So the 28 documents were pre-split into 263 section-level chunks *before* upload.
That's why the counts don't match. And that's a workshop convenience: in a real
build the split belongs **inside** this pipeline, as another Beam stage between
read and embed — so that when someone drops a new document in the bucket it gets
chunked automatically, instead of depending on someone remembering to run a
script.

That's the second pipeline assumption you've hit today. The external table assumed
one record per line. This one assumes one file is one sensible unit of meaning.
Neither checks. Neither warns.

**Verify**, in Cloud SQL Studio:

```sql
SELECT COUNT(*) FROM policy_chunks;   -- expect 263, + the 1 row you inserted by hand
```

### The RAG agent

Open `~/everstorm-dataengineer/support_agent/agent.py` and fill in the three
markers below. `solutions/agent_completed.py` has the finished version if you get
stuck.

**`#REPLACE RAG-CONVERT EMBEDDING`:**

```python
result = client.models.embed_content(
    model="text-embedding-005",
    contents=question,
    config=EmbedContentConfig(
        task_type="RETRIEVAL_QUERY",   # query side, not RETRIEVAL_DOCUMENT
        output_dimensionality=768,
    ),
)
```

⚠️ **This is the asymmetry from Part 1**, and it's the one the starter code gets
wrong. `RETRIEVAL_QUERY` here, because this is a question; `RETRIEVAL_DOCUMENT`
when the corpus was embedded, because those were documents.

Using `RETRIEVAL_DOCUMENT` on both sides still returns results — the vectors stay
in one space — which is exactly why this is easy to get wrong and never notice.

**`#REPLACE RAG-RETRIEVE`:**

```python
cursor.execute(
    "SELECT chunk_content FROM policy_chunks ORDER BY embedding <=> %s LIMIT 5",
    ([query_embedding]),
)
```

**`#REPLACE-CALL RAG`** — the agent itself:

```python
root_agent = LlmAgent(
    model="gemini-2.5-flash",
    name="everstorm_support_agent",
    instruction="""
        You are a senior customer-support advisor for Everstorm Outfitters, an
        outdoor gear retailer. You answer using the company's own policy
        documents, not from general knowledge about retail.

        **Your process:**
        1. ALWAYS call `policy_lookup` first, before answering anything.
        2. Answer ONLY from what it returns. Quote the specific figure, date or
           condition rather than paraphrasing it away.
        3. Every retrieved passage begins with a context line naming the
           document, its revision and its status. USE IT:
           - If a passage is marked `Status: superseded`, do NOT use it for a
             current question. Prefer the `Status: current` passage. Mention the
             older value only if the customer asked what the policy used to be.
           - If passages disagree, say so explicitly, name both documents, and
             state which one governs.
        4. Regional rules are not interchangeable. The EU and the UK are treated
           differently on duties. Never apply an EU rule to a UK order.
        5. If the retrieved passages do not contain the answer, say you do not
           know and name the team to contact. NEVER invent a policy, a number,
           or a date.

        **Output format:** a direct answer first, then the specific conditions
        or exceptions, then the document you relied on.
    """,
    tools=[policy_lookup],
)
```

💡 Steps 3 and 4 are doing the job the database schema can't — they're how the
model compensates for there being no `status` or `region` column to filter on.

**Run it locally**, in a Cloud Shell *terminal* — `adk run` is interactive, so it
won't work from a notebook cell:

```bash
cd ~/everstorm-dataengineer/
. ~/everstorm-dataengineer/set_env.sh
source ~/everstorm-dataengineer/env/bin/activate
pip install -r support_agent/requirements.txt
adk run support_agent
```

### Ask these three, in this order

Each one comes from `eval/rag_eval.jsonl` and each is chosen to show exactly one
thing.

**1. `How long do I have to return something?` — the baseline.**

💡 **Watch the logs, not the answer.** It called the tool *first*. Before saying
anything it searched the database, pulled back five chunks, and only then started
writing.

That ordering is the entire thing. That's RAG. The model isn't answering from what
it absorbed in training — it knows nothing about Everstorm, the company doesn't
exist. It's answering from documents you handed it thirty seconds ago. And that's
why you can point at a source. Ask a plain chatbot about a returns policy and you
get something plausible; ask this and you get something you can go and check.

**2. `What's the free shipping threshold?` — the version trap.**

💡 There are two shipping policies in that knowledge base. The current one says
free shipping over **$75**. An old one, revision 3, retired last year, says
**$99**. Both are in there. Both are about shipping thresholds. To a similarity
search they look almost identical — that's the problem. **Semantic search has no
idea what "current" means. It only knows what's close.**

The only thing standing between you and a wrong answer is that context line baked
into every chunk, plus the instruction telling the model to read it. A good answer
gives $75 *and* flags that the older figure exists and is retired — because a
customer holding a printout from last year deserves that sentence.

**3. `I'm in the UK, do I pay import duty?` — the regional trap.**

💡 The EU ships duties-**paid**. The UK ships duties-**unpaid**. Completely
different answers. And they live in the same document, with both regions in its
title.

So if retrieval grabs the EU paragraph — which is right there, textually adjacent,
semantically almost identical — the model will tell a British customer they owe
nothing. Confidently. With a citation.

⚠️ **That's the dangerous class of RAG error.** Not "I don't know". A fluent,
well-sourced, *wrong* answer.

### If one of them fails, don't move on — diagnose it

💡 There are exactly **two** possibilities:

1. **Retrieval failed** — you handed the model the wrong chunks, and it did fine
   with bad input.
2. **Retrieval was fine** — you gave it the right chunks and it misread them.

Completely different fixes. Bad retrieval means chunking, or embedding, or
top-*k*. Bad reading means the instruction prompt.

So look at what actually came back before you change anything. Print the retrieved
chunks and read them against the question.

A demo that always works teaches nobody how to debug one. Even if all three pass,
spend thirty seconds working out what you *would* have checked — it's the most
transferable minute in the workshop.

### Deploy to Cloud Run

⏳ Build + deploy takes 5–8 minutes.

In [ ]:
%%bash
. ~/everstorm-dataengineer/set_env.sh
cd ~/everstorm-dataengineer/
gcloud builds submit . --project=${PROJECT_ID} --region=${REGION} \
  --substitutions=_AGENT_NAME=${AGENT_NAME},_IMAGE_PATH=${IMAGE_PATH}

gcloud run deploy ${SERVICE_NAME} \
  --image=${IMAGE_PATH} --platform=managed --region=${REGION} \
  --set-env-vars="A2A_HOST=0.0.0.0,A2A_PORT=8080" \
  --set-env-vars="GOOGLE_GENAI_USE_VERTEXAI=TRUE" \
  --set-env-vars="GOOGLE_CLOUD_LOCATION=${REGION}" \
  --set-env-vars="GOOGLE_CLOUD_PROJECT=${PROJECT_ID}" \
  --set-env-vars="PROJECT_ID=${PROJECT_ID},PUBLIC_URL=${PUBLIC_URL},REGION=${REGION}" \
  --set-env-vars="INSTANCE_NAME=${INSTANCE_NAME}" \
  --set-env-vars="DB_USER=${DB_USER},DB_PASSWORD=${DB_PASSWORD},DB_NAME=${DB_NAME}" \
  --allow-unauthenticated --project=${PROJECT_ID} --min-instances=1

💡 **Why A2A, and not a library.** This isn't deployed as a package something
imports. It goes out as its own HTTP service, with an **agent card**: a
machine-readable description saying "here's my name, here's what I'm good at,
here's how to call me."

That's the A2A pattern — agent to agent — and the reason it matters is
organisational, not technical. If every agent in your company lives in one
runtime, then one team's deploy is everybody's deploy, and one team's memory leak
is everybody's outage. This way the support team owns this service, ships it on
its own schedule, and scales it on its own traffic. Somebody else's agent
discovers it and calls it over HTTP. You pay a network hop and you buy back
independence.

💡 **`--min-instances=1`** keeps a container warm so the first question of the
morning isn't a ten-second cold start. You're paying for an idle instance to buy
latency. That's a real trade — make it deliberately.

### ⚠️ The hardening checklist — don't skip this

The **architecture** here is genuinely production-shaped: managed services,
containerised, scales horizontally, service accounts rather than user
credentials. That part is real.

But there are four things in what you just ran that should not go in front of
customers:

1. **`DB_PASSWORD` in plain text**, as an environment variable on the deploy
   command — and the default in this repo is `1234qwer`. That belongs in **Secret
   Manager**, referenced with `--set-secrets`. Better still, switch the connector
   to **IAM database authentication** and have no password at all.
2. **`--allow-unauthenticated`.** Anybody on the internet who finds this URL can
   call your agent, and every call costs you Gemini tokens. Fine for a workshop.
   A real deployment wants IAM-authenticated invocation, or internal-only ingress.
3. **The Cloud SQL instance is single-zone.** One zone goes down and your
   knowledge base is gone. Production wants a regional configuration.
4. **There's no test stage anywhere in that build.** It compiles, it ships.
   Nothing checks that the agent still answers correctly before it goes live —
   which, for a system whose entire job is factual accuracy, is the gap to close
   first.

None of that makes today's build wrong. It makes it a **workshop build**. Knowing
precisely which corners are cut is the difference between shipping this and
shipping something that looks like this.

### What you built

Two piles of English that nobody could query.

**180 support tickets.** Which warehouse was slowest, which product was failing —
nobody knew. Now they're three tables and the answers are a `GROUP BY` away.
Neither had ever been written down.

**28 policy documents.** Now there's a URL. Ask it about UK import duty and it
answers from the actual policy, and tells you which document it used.

💡 **One architecture, two halves, and the split is the thing to keep if you
forget everything else: BigQuery for _building_ knowledge — big, slow, analytical,
batch. Cloud SQL for _serving_ it — small, fast, operational, one question at a
time.** Same three ideas on both sides: chunk, embed, search. Different engine,
because it's a different job.

And the honest thread running through all of it: **every pipeline you touched
carried an assumption about the shape of its input.** One record per line. One
file, one unit of meaning. None of them checked, none of them warned, and all of
them would have quietly handed you something that looked like data.

That's the job. The models are the easy part now.

## 5. _Optional_ — score the eval set

Three questions answered correctly is an **anecdote**. It is not evidence. This is
the part that separates a demo from a system, and it takes about five minutes.

`eval/rag_eval.jsonl` holds **32 questions** with known answers and known source
documents — each with `ground_truth_docs`, optional `distractor_docs`, a `type`,
and a note on the intended failure mode.

**Score two things separately**, because they fail for different reasons:

- **Retrieval** — did `ground_truth_docs` appear in the top-*k*? If this is low,
  the problem is chunking, or embedding, or top-*k*.
- **Generation** — given the right documents, did the answer match
  `expected_answer`? If retrieval is high and generation is low, the documents
  were fine and the **instruction prompt** is wrong.

💡 One number tells you nothing. Two numbers tell you where to go next.

If you're short on time, the `version_conflict` and `regional_disambiguation`
questions alone make the point. If you have room, run the two `unanswerable` ones
and watch it decline to answer — **abstention is a result**, and a system that
never says "I don't know" is a system that's lying somewhere.

💡 **This is what belongs in CI.** Nothing in the build pipeline you just ran
checks whether the agent still answers correctly before it ships. For a system
whose whole job is factual accuracy, that's the gap to close first.

`eval/sql_eval.md` does the equivalent for the Part 1 tables.

## 6. Design decisions, and the reasoning behind each

If you're wondering "why is it done this way", these are the eight answers. Each
is a real constraint, not a preference.

| # | Decision | Why |
|---|---|---|
| 1 | Tickets flattened to one line per file | The `§`-delimited external table reads one row per line; multi-line files give 3,100 fragments instead of 180 tickets |
| 2 | `quote = ''` in the external table OPTIONS | Default CSV quote handling breaks on ordinary prose punctuation |
| 3 | KB pre-split into 263 section chunks | The pipeline has no chunking stage; one 1,000-word doc per vector retrieves poorly |
| 4 | Doc title/revision/status baked into chunk text | The Cloud SQL schema has no metadata columns, so this is the only way version info survives retrieval |
| 5 | Extraction asks for `closed_date`, not `resolution_days` | The prose states the date; days are derived in SQL, where it's deterministic |
| 6 | `issue_category` given an explicit enum | It's never stated literally and must be classified; unconstrained, labels fragment |
| 7 | Query-side embedding uses `RETRIEVAL_QUERY` | Questions and documents are asymmetric; matching task types is the correct pairing |
| 8 | The workshop ends on a scored eval run | A measured retrieval/generation score is better evidence than a working anecdote, and it's what belongs in CI |

Decisions 1, 3 and 4 are all the same lesson — **a pipeline carries assumptions
about the shape of its input** — and that lesson is more durable than any single
command here.

## 7. Cleanup — do this when you finish

⚠️ **Run this.** The Cloud SQL instance is the expensive one, and it bills for
every hour it exists whether you use it or not. This is what keeps the whole
workshop inside a few dollars of free credit.

```bash
. ~/everstorm-dataengineer/set_env.sh
gcloud run services delete ${SERVICE_NAME} --region=${REGION} --quiet
gcloud artifacts repositories delete ${REPO_NAME} --location=${REGION} --quiet
bq rm -r -f --dataset ${PROJECT_ID}:everstorm_data
bq rm --force --connection --project_id=${PROJECT_ID} --location=${REGION} gcs-connection
gcloud sql instances delete ${INSTANCE_NAME} --project=${PROJECT_ID} --quiet
gcloud storage rm -r gs://${BUCKET_NAME} --quiet
rm -rf ~/everstorm-dataengineer ~/project_id.txt
```

Then confirm the expensive one actually went:

```bash
gcloud sql instances list      # should be empty
```